# repositorio-sincronizar.ipynb — traz o GitHub pro Drive

O código vive no **GitHub**; o Colab lê do **Drive**. Este notebook é a ponte:
clona o repositório e copia `modulos/`, `notebooks/` e `dados_lexico/` por cima
do que está no Drive.

Rode **sempre que o repositório mudar** — e desconfie se fizer semanas que você
não roda.

## Por que isso precisa existir

Em 29/ago o Drive estava **56 commits atrás**: faltavam 10 notebooks (o portão
de qualidade, a compilação, os dois da Bíblia, os quatro do chinês) e 7
módulos, e os que existiam eram de 19–23/ago. Um teste rodado assim executa
código velho e falha por motivo que não existe mais no repositório — o pior
tipo de depuração.

## Por que copiar por CIMA, e não apagar e recriar

`cp` por cima escreve **no mesmo arquivo** do Drive: o id não muda, e o link do
Colab que você tem salvo continua abrindo o notebook certo. Apagar e recriar
daria um arquivo novo, com id novo, e todos os seus links quebrariam de uma vez.

## O que este notebook NÃO toca

Só `pipeline/`. Nada de `videos/`, `assets/` ou planilha — mídia e estoque não
estão no git e não têm o que sincronizar. A direção é **uma só**: GitHub → Drive.
Se você editou um notebook direto no Colab e não levou pro git, essa edição é
sobrescrita — a célula 3 te mostra o que vai mudar antes de mudar.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP                                                         ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import shutil, subprocess, hashlib
from pathlib import Path

REPO = "https://github.com/alanabdmorais/narrated_video"
CLONE = Path("/content/repo_narrated_video")

if CLONE.exists():
    shutil.rmtree(CLONE)
print(f"⬇️  clonando {REPO}")
subprocess.run(["git", "clone", "--depth", "1", "--quiet", REPO, str(CLONE)], check=True)

commit = subprocess.run(["git", "-C", str(CLONE), "log", "-1", "--format=%h %cs %s"],
                        capture_output=True, text=True).stdout.strip()
print(f"✅ {commit}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO — edite só esta célula                          ║
# ╚══════════════════════════════════════════════════════════════════╝

PASTA_DRIVE_RAIZ = "narrated_video"     # ⚠️ a mesma dos outros notebooks

# O que sincronizar. Tudo sob pipeline/ — o resto (videos/, assets/, planilhas)
# não está no git e não tem o que sincronizar.
PASTAS = ["modulos", "notebooks", "dados_lexico"]

DESTINO = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline")
ORIGEM  = CLONE / "pipeline"

print(f"de   {ORIGEM}")
print(f"para {DESTINO}")
print(f"pastas: {', '.join(PASTAS)}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 O QUE VAI MUDAR — confira ANTES de copiar                     ║
# ╚══════════════════════════════════════════════════════════════════╝
# Nada é escrito nesta célula. Ela existe pra você ver, antes, se algum
# arquivo que você editou direto no Colab está prestes a ser sobrescrito.

def _sha(caminho):
    return hashlib.sha256(caminho.read_bytes()).hexdigest()

plano = {"novo": [], "muda": [], "igual": [], "so_no_drive": []}

for pasta in PASTAS:
    org, dst = ORIGEM / pasta, DESTINO / pasta
    if not org.exists():
        print(f"⚠️  {pasta}/ não existe no repositório — pulando")
        continue
    no_repo = {p.name for p in org.iterdir() if p.is_file()}
    for nome in sorted(no_repo):
        a, b = org / nome, dst / nome
        if not b.exists():
            plano["novo"].append(f"{pasta}/{nome}")
        elif _sha(a) != _sha(b):
            plano["muda"].append(f"{pasta}/{nome}")
        else:
            plano["igual"].append(f"{pasta}/{nome}")
    if dst.exists():
        for p in sorted(dst.iterdir()):
            if p.is_file() and p.name not in no_repo and not p.name.startswith("."):
                plano["so_no_drive"].append(f"{pasta}/{p.name}")

print(f"➕ {len(plano['novo'])} novo(s) — não existem no Drive ainda")
for f in plano["novo"]:  print(f"     {f}")
print(f"\n♻️  {len(plano['muda'])} vai(vão) ser sobrescrito(s) pelo repositório")
for f in plano["muda"]:  print(f"     {f}")
print(f"\n✅ {len(plano['igual'])} já idêntico(s) — serão pulados")
print(f"\n👻 {len(plano['so_no_drive'])} só no Drive — NÃO serão apagados")
for f in plano["so_no_drive"]: print(f"     {f}")
if plano["so_no_drive"]:
    print("\n   (arquivo que existe só no Drive é ou lixo de uma versão antiga,")
    print("    ou algo que nunca foi pro git. Este notebook não apaga nada —")
    print("    confira e apague à mão se for lixo.)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 COPIAR — só rode depois de conferir a célula acima            ║
# ╚══════════════════════════════════════════════════════════════════╝

copiados = 0
for caminho in plano["novo"] + plano["muda"]:
    pasta, nome = caminho.split("/", 1)
    (DESTINO / pasta).mkdir(parents=True, exist_ok=True)
    # copyfile (e não copy2) escreve NO MESMO arquivo do Drive quando ele já
    # existe -- o id não muda, e o link do Colab que você tem salvo continua
    # valendo. Apagar e recriar quebraria todos os links de uma vez.
    shutil.copyfile(ORIGEM / pasta / nome, DESTINO / pasta / nome)
    copiados += 1
    print(f"   {caminho}")

print(f"\n✅ {copiados} arquivo(s) sincronizado(s) ({len(plano['igual'])} já estavam iguais)")
print(f"   Drive agora em: {commit}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ✅ CONFERIR — o Drive ficou byte a byte igual ao repositório?     ║
# ╚══════════════════════════════════════════════════════════════════╝
# Copiar pra um Drive montado pode falhar por cota, sessão expirada ou
# arquivo aberto -- e o shutil nem sempre grita. Aqui a prova é o hash.

divergentes = []
for pasta in PASTAS:
    org = ORIGEM / pasta
    if not org.exists():
        continue
    for a in sorted(p for p in org.iterdir() if p.is_file()):
        b = DESTINO / pasta / a.name
        if not b.exists() or _sha(a) != _sha(b):
            divergentes.append(f"{pasta}/{a.name}")

if divergentes:
    print(f"❌ {len(divergentes)} arquivo(s) NÃO bateram — rode a célula de copiar de novo:")
    for f in divergentes:
        print(f"     {f}")
else:
    print("✅ Drive idêntico ao repositório, byte a byte.")
    print(f"   {commit}")